# Notebook 3: The Transformer Architecture — From Scratch

## What You'll Learn
- Self-attention: the mechanism that powers every modern LLM
- Build self-attention, multi-head attention, and full Transformer blocks from scratch
- Understand masked (causal) attention for text generation
- Train a small GPT-style model on real text

---

## The Paper That Changed Everything

The Transformer was introduced in ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) (Vaswani et al., 2017). Before this, language models used RNNs/LSTMs which processed text **sequentially** (one word at a time). The Transformer processes all tokens **in parallel** using attention.

Every modern LLM — GPT-4, Claude, LLaMA, Gemini — is a Transformer.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Self-Attention: The Core Idea

### The Problem
Consider the sentence: *"The animal didn't cross the street because **it** was too tired."*

What does "it" refer to? The **animal**, not the street. A model needs to connect "it" back to "animal" across a distance of 5 tokens.

### The Solution: Attention

For each token, attention asks: **"Which other tokens should I pay attention to?"**

The mechanism:
1. Each token produces three vectors: **Query (Q)**, **Key (K)**, **Value (V)**
2. **Query**: "What am I looking for?"
3. **Key**: "What do I contain?"
4. **Value**: "What information do I provide?"

The attention score between tokens $i$ and $j$ is:

$$\text{score}(i, j) = \frac{Q_i \cdot K_j}{\sqrt{d_k}}$$

Then we softmax and take weighted average of Values:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [ ]:
# Let's build self-attention step by step

# Suppose we have a sequence of 4 tokens, each with embedding dim 8
seq_len = 4
d_model = 8
d_k = 8  # dimension of Q, K, V

# Random token embeddings (normally these come from the embedding layer)
X = torch.randn(seq_len, d_model)
print(f'Input X shape: {X.shape}  (seq_len, d_model)')
print(f'X =\n{X}')

In [ ]:
# Step 1: Create Q, K, V projection matrices
# These are learned parameters — the model learns WHAT to attend to

W_Q = torch.randn(d_model, d_k)  # Query projection
W_K = torch.randn(d_model, d_k)  # Key projection  
W_V = torch.randn(d_model, d_k)  # Value projection

# Step 2: Compute Q, K, V for each token
Q = X @ W_Q  # (seq_len, d_k)
K = X @ W_K  # (seq_len, d_k)
V = X @ W_V  # (seq_len, d_k)

print(f'Q shape: {Q.shape}')
print(f'K shape: {K.shape}')
print(f'V shape: {V.shape}')

In [ ]:
# Step 3: Compute attention scores
# Score = Q @ K^T / sqrt(d_k)

scores = Q @ K.T / math.sqrt(d_k)
print(f'Attention scores shape: {scores.shape}  (seq_len x seq_len)')
print(f'\nRaw scores (each row = one token\'s attention to all others):')
print(scores)

# Step 4: Apply softmax to get attention weights (probabilities)
attention_weights = F.softmax(scores, dim=-1)
print(f'\nAttention weights (each row sums to 1.0):')
print(attention_weights)
print(f'\nRow sums: {attention_weights.sum(dim=-1)}')

In [ ]:
# Step 5: Weighted sum of Values
output = attention_weights @ V
print(f'Output shape: {output.shape}  (same as input!)')
print(f'\nEach token\'s output is a weighted combination of all Value vectors.')
print(f'The weights are the attention scores.')

In [ ]:
# Visualize the attention weights
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(attention_weights.detach().numpy(), cmap='Blues', vmin=0, vmax=1)

ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels([f'Token {i}' for i in range(seq_len)])
ax.set_yticklabels([f'Token {i}' for i in range(seq_len)])
ax.set_xlabel('Key (attending TO)')
ax.set_ylabel('Query (attending FROM)')
ax.set_title('Self-Attention Weights')

for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{attention_weights[i,j]:.2f}', ha='center', va='center', fontsize=12)

plt.colorbar(im)
plt.tight_layout()
plt.show()

## 2. Causal (Masked) Self-Attention

For **text generation**, there's a crucial constraint: token $i$ should **only attend to tokens before it** (positions $0, 1, \ldots, i$). This is because during generation, we don't have future tokens yet.

This is called **causal masking** or **masked self-attention**.

We implement it by setting future positions to $-\infty$ before the softmax:

$$\text{mask}[i][j] = \begin{cases} 0 & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

In [ ]:
# Create a causal mask
def create_causal_mask(seq_len):
    """Lower triangular mask: position i can attend to positions 0..i"""
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    return mask  # True = MASKED (cannot attend)

mask = create_causal_mask(seq_len)
print("Causal mask (True = blocked):")
print(mask.int())
print("\nToken 0 can only see: itself")
print("Token 1 can see: tokens 0-1")
print("Token 2 can see: tokens 0-2")
print("Token 3 can see: tokens 0-3 (all)")

In [ ]:
# Apply the mask to attention scores
masked_scores = scores.clone()
masked_scores[mask] = float('-inf')  # Set future positions to -inf

print("Masked scores:")
print(masked_scores)

# Apply softmax — the -inf positions become 0 probability
causal_attention_weights = F.softmax(masked_scores, dim=-1)
print("\nCausal attention weights:")
print(causal_attention_weights)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, weights, title in [
    (ax1, attention_weights, 'Full Self-Attention'),
    (ax2, causal_attention_weights, 'Causal (Masked) Self-Attention')
]:
    im = ax.imshow(weights.detach().numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels([f'Tok {i}' for i in range(seq_len)])
    ax.set_yticklabels([f'Tok {i}' for i in range(seq_len)])
    ax.set_title(title)
    for i in range(seq_len):
        for j in range(seq_len):
            ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=11)

plt.tight_layout()
plt.show()

print("Causal attention: each token can only attend to previous tokens (and itself).")
print("This is what makes autoregressive generation possible.")

## 3. Multi-Head Attention

One attention head captures one type of relationship (maybe syntactic). But we want to capture **multiple** types of relationships simultaneously.

**Multi-head attention** runs $h$ attention heads in parallel, each with its own $W_Q, W_K, W_V$ projections, then concatenates the results.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

Where each head has reduced dimension: $d_k = d_{\text{model}} / h$

Different heads learn to attend to different things:
- Head 1 might learn syntactic relationships (subject-verb)
- Head 2 might learn coreference (pronoun resolution)
- Head 3 might learn positional patterns

In [ ]:
class MultiHeadCausalAttention(nn.Module):
    """Multi-head causal self-attention — the core of GPT."""
    
    def __init__(self, d_model, num_heads, max_seq_len=512, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # Q, K, V projections (combined for efficiency)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        # Output projection
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
        # Pre-compute causal mask
        mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        self.register_buffer('mask', mask)
    
    def forward(self, x, return_attention=False):
        """
        x: (batch_size, seq_len, d_model)
        returns: (batch_size, seq_len, d_model)
        """
        B, T, C = x.shape
        
        # Compute Q, K, V for all heads at once
        qkv = self.qkv_proj(x)  # (B, T, 3 * d_model)
        Q, K, V = qkv.chunk(3, dim=-1)  # Each: (B, T, d_model)
        
        # Reshape for multi-head: (B, T, d_model) -> (B, num_heads, T, head_dim)
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Compute attention scores
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, H, T, T)
        
        # Apply causal mask
        scores = scores.masked_fill(self.mask[:T, :T], float('-inf'))
        
        # Softmax and dropout
        attn_weights = F.softmax(scores, dim=-1)  # (B, H, T, T)
        attn_weights = self.dropout(attn_weights)
        
        # Weighted sum of values
        out = attn_weights @ V  # (B, H, T, head_dim)
        
        # Reshape back: (B, H, T, head_dim) -> (B, T, d_model)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        
        # Final projection
        out = self.out_proj(out)
        
        if return_attention:
            return out, attn_weights
        return out

# Test it
d_model = 64
num_heads = 4
mha = MultiHeadCausalAttention(d_model, num_heads)

x = torch.randn(2, 10, d_model)  # batch=2, seq_len=10
output, attn = mha(x, return_attention=True)

print(f'Input shape:  {x.shape}')
print(f'Output shape: {output.shape}')
print(f'Attention weights shape: {attn.shape}  (batch, heads, seq_len, seq_len)')
print(f'\nParameters: {sum(p.numel() for p in mha.parameters()):,}')

In [ ]:
# Visualize attention patterns for each head
fig, axes = plt.subplots(1, num_heads, figsize=(16, 4))

for head in range(num_heads):
    ax = axes[head]
    weights = attn[0, head].detach().numpy()  # first example, head i
    im = ax.imshow(weights, cmap='Blues', vmin=0)
    ax.set_title(f'Head {head+1}', fontsize=12)
    ax.set_xlabel('Key position')
    if head == 0:
        ax.set_ylabel('Query position')

plt.suptitle('Attention Patterns Per Head (randomly initialized)', fontsize=14)
plt.tight_layout()
plt.show()

print("After training, each head learns different attention patterns.")
print("Some heads learn local patterns, some learn long-range dependencies.")

## 4. The Full Transformer Block

A Transformer block combines:
1. **Multi-Head Self-Attention** (communication between tokens)
2. **Feed-Forward Network** (processing within each token independently)
3. **Layer Normalization** (stabilizes training)
4. **Residual Connections** (helps gradient flow)

```
Input
  │
  ├──→ LayerNorm → Multi-Head Attention ──→ (+)
  │                                          │
  └──────────────── (residual) ──────────────┘
                                              │
  ├──→ LayerNorm → Feed-Forward Network ──→ (+)
  │                                          │
  └──────────────── (residual) ──────────────┘
                                              │
                                           Output
```

In [ ]:
class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network.
    
    Applied independently to each position (token).
    This is where the model "thinks" about individual tokens.
    Attention handles communication; FFN handles computation.
    """
    
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        if d_ff is None:
            d_ff = 4 * d_model  # Standard ratio in Transformers
        
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),  # Modern LLMs use GELU instead of ReLU
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """A single Transformer block — the building block of GPT."""
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        
        # Layer norms (Pre-LN variant, used in GPT-2 and later)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        # Multi-head attention
        self.attention = MultiHeadCausalAttention(d_model, num_heads, dropout=dropout)
        
        # Feed-forward network
        self.ffn = FeedForward(d_model, dropout=dropout)
    
    def forward(self, x):
        # Pre-LN: normalize BEFORE the sublayer
        # Residual connection: add input back to output
        x = x + self.attention(self.ln1(x))  # Attention + residual
        x = x + self.ffn(self.ln2(x))        # FFN + residual
        return x

# Test a single Transformer block
block = TransformerBlock(d_model=64, num_heads=4)
x = torch.randn(2, 10, 64)
output = block(x)

print(f'Input shape:  {x.shape}')
print(f'Output shape: {output.shape}')
print(f'Parameters:   {sum(p.numel() for p in block.parameters()):,}')

## 5. Building a Complete GPT Model

Now let's stack everything together into a complete GPT-style language model:

1. **Token Embeddings** + **Positional Embeddings** (from Notebook 2)
2. **N Transformer Blocks** stacked on top of each other
3. **Final Layer Norm**
4. **Linear head** to project back to vocabulary logits

In [ ]:
class MiniGPT(nn.Module):
    """A minimal GPT-style language model."""
    
    def __init__(self, vocab_size, d_model, num_heads, num_layers, 
                 max_seq_len=512, dropout=0.1):
        super().__init__()
        
        self.max_seq_len = max_seq_len
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])
        
        # Final layer norm and output projection
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying: share weights between token embedding and output
        # This is a standard trick that improves performance
        self.token_embedding.weight = self.lm_head.weight
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, token_ids, targets=None):
        """
        token_ids: (batch_size, seq_len)
        targets: (batch_size, seq_len) — optional, for computing loss
        """
        B, T = token_ids.shape
        
        # Embeddings
        tok_emb = self.token_embedding(token_ids)  # (B, T, d_model)
        pos_emb = self.position_embedding(
            torch.arange(T, device=token_ids.device)  # (T,)
        )  # (T, d_model)
        x = self.dropout(tok_emb + pos_emb)  # (B, T, d_model)
        
        # Pass through Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Final layer norm and project to vocabulary
        x = self.ln_final(x)  # (B, T, d_model)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        # Compute loss if targets provided
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),  # (B*T, vocab_size)
                targets.view(-1)  # (B*T,)
            )
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, start_tokens, max_new_tokens=100, temperature=1.0, top_k=None):
        """
        Autoregressive text generation.
        start_tokens: (1, seq_len) — prompt tokens
        """
        self.eval()
        tokens = start_tokens.clone()
        
        for _ in range(max_new_tokens):
            # Crop to max_seq_len
            context = tokens[:, -self.max_seq_len:]
            
            # Forward pass
            logits, _ = self(context)
            logits = logits[:, -1, :] / temperature  # Last position only
            
            # Optional: top-k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            # Sample from the distribution
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # (B, 1)
            
            tokens = torch.cat([tokens, next_token], dim=1)
        
        self.train()
        return tokens

# Show model architecture
print("MiniGPT Architecture:")
print("="*50)

In [ ]:
# Compare model sizes

configs = [
    {"name": "Tiny",   "d_model": 64,  "heads": 4, "layers": 2},
    {"name": "Small",  "d_model": 128, "heads": 4, "layers": 4},
    {"name": "Medium", "d_model": 256, "heads": 8, "layers": 6},
    {"name": "Large",  "d_model": 512, "heads": 8, "layers": 8},
]

print(f'{"Config":<10} {"d_model":<10} {"Heads":<8} {"Layers":<8} {"Parameters":<15}')
print("-" * 55)

for cfg in configs:
    m = MiniGPT(
        vocab_size=256,  # byte-level for now
        d_model=cfg["d_model"],
        num_heads=cfg["heads"],
        num_layers=cfg["layers"],
    )
    params = sum(p.numel() for p in m.parameters())
    print(f'{cfg["name"]:<10} {cfg["d_model"]:<10} {cfg["heads"]:<8} {cfg["layers"]:<8} {params:>12,}')

print()
print("For reference:")
print("  GPT-2 Small:  124M params (d=768, 12 heads, 12 layers)")
print("  GPT-2 Large:  774M params (d=1280, 20 heads, 36 layers)")
print("  GPT-3:        175B params (d=12288, 96 heads, 96 layers)")
print("  LLaMA-2 7B:   7B params (d=4096, 32 heads, 32 layers)")

## 6. Training Our MiniGPT

Let's train our model on real text! We'll use Shakespeare for a classic demonstration.

In [ ]:
# Prepare training data

text = """HAMLET: To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles,
And by opposing end them. To die, to sleep;
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to, 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep: perchance to dream: ay, there's the rub;
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil,
Must give us pause. There's the respect
That makes calamity of so long life;
For who would bear the whips and scorns of time,
The oppressor's wrong, the proud man's contumely,
The pangs of despised love, the law's delay,
The insolence of office and the spurns
That patient merit of the unworthy takes,
When he himself might his quietus make
With a bare bodkin? Who would fardels bear,
To grunt and sweat under a weary life,
But that the dread of something after death,
The undiscoverd country from whose bourn
No traveller returns, puzzles the will
And makes us rather bear those ills we have
Than fly to others that we know not of?
Thus conscience does make cowards of us all;
And thus the native hue of resolution
Is sicklied o'er with the pale cast of thought,
And enterprises of great pitch and moment
With this regard their currents turn awry,
And lose the name of action. Soft you now!
The fair Ophelia! Nymph, in thy prayers
Be all my sins remember'd.

OPHELIA: Good my lord,
How does your honour for this many a day?

HAMLET: I humbly thank you; well, well, well.

OPHELIA: My lord, I have remembrances of yours,
That I have longed long to re-deliver;
I pray you, now receive them.

HAMLET: No, not I; I never gave you aught.

OPHELIA: My honour'd lord, you know right well you did;
And, with them, words of so sweet breath composed
As made the things more rich: their perfume lost,
Take these again; for to the noble mind
Rich gifts wax poor when givers prove unkind.
There, my lord."""

# Character-level encoding (byte-level)
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
encode_fn = lambda s: [char_to_idx[c] for c in s]
decode_fn = lambda ids: ''.join([idx_to_char[i] for i in ids])

data = torch.tensor(encode_fn(text), dtype=torch.long)
print(f'Text length: {len(text)} characters')
print(f'Vocabulary size: {vocab_size}')
print(f'Data tensor shape: {data.shape}')

In [ ]:
# Create training batches

block_size = 64  # Context window size
batch_size = 16

def get_batch(data, block_size, batch_size):
    """Generate a random batch of training data."""
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# Test
xb, yb = get_batch(data, block_size, batch_size)
print(f'Input batch shape:  {xb.shape}  (batch_size, block_size)')
print(f'Target batch shape: {yb.shape}  (batch_size, block_size)')
print(f'\nExample input:  "{decode_fn(xb[0].tolist())}"')
print(f'Example target: "{decode_fn(yb[0].tolist())}"')
print(f'(Target is shifted by 1 position)')

In [ ]:
# Create and train MiniGPT

model = MiniGPT(
    vocab_size=vocab_size,
    d_model=64,
    num_heads=4,
    num_layers=4,
    max_seq_len=block_size,
    dropout=0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'MiniGPT total parameters: {total_params:,}')
print(f'\nModel architecture:')
print(model)

In [ ]:
# Training loop

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
losses = []
num_steps = 2000

for step in range(num_steps):
    # Get batch
    xb, yb = get_batch(data, block_size, batch_size)
    xb, yb = xb.to(device), yb.to(device)
    
    # Forward pass
    logits, loss = model(xb, targets=yb)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    # Gradient clipping (prevents exploding gradients)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    losses.append(loss.item())
    
    if (step + 1) % 400 == 0:
        avg_loss = np.mean(losses[-100:])
        print(f'Step {step+1:5d} | Loss: {avg_loss:.4f}')

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.3, label='Raw loss')
# Smoothed
window = 50
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(losses)), smoothed, label=f'Smoothed ({window}-step)', linewidth=2)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('MiniGPT Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Generate text!

prompts = ["HAMLET:", "To be", "The ", "OPHELIA:"]

for prompt in prompts:
    start_tokens = torch.tensor([encode_fn(prompt)], dtype=torch.long).to(device)
    generated = model.generate(start_tokens, max_new_tokens=150, temperature=0.8, top_k=10)
    generated_text = decode_fn(generated[0].tolist())
    
    print(f'\n{"="*60}')
    print(f'Prompt: "{prompt}"')
    print(f'{"="*60}')
    print(generated_text)

In [ ]:
# Compare different temperature settings

prompt = "HAMLET: "
start_tokens = torch.tensor([encode_fn(prompt)], dtype=torch.long).to(device)

temperatures = [0.3, 0.7, 1.0, 1.5]

for temp in temperatures:
    generated = model.generate(start_tokens, max_new_tokens=100, temperature=temp)
    generated_text = decode_fn(generated[0].tolist())
    
    print(f'\nTemperature = {temp}')
    print(f'  {generated_text[:120]}...')

print("\n\nLow temp → more repetitive but coherent")
print("High temp → more creative but potentially nonsensical")

## 7. Exercises

### Exercise 1: Scale It Up
Download the full Tiny Shakespeare dataset and train a larger model. How does quality improve with:
- More layers (6, 8, 12)?
- Larger d_model (128, 256)?
- More training steps?

In [ ]:
# EXERCISE 1: Scale up
# Download data:
# import urllib.request
# url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
# urllib.request.urlretrieve(url, 'shakespeare.txt')

# Your code here

### Exercise 2: Attention Visualization
Modify the model to return attention weights from all layers and heads. Visualize them for a given input sentence. Can you identify what patterns different heads learn?

In [ ]:
# EXERCISE 2: Visualize learned attention patterns
# Your code here

### Exercise 3: Learning Rate Scheduling
Implement the "warmup + cosine decay" learning rate schedule used by real LLMs:

$$\text{lr}(t) = \begin{cases} \text{lr}_{\max} \cdot \frac{t}{T_{\text{warmup}}} & \text{if } t < T_{\text{warmup}} \\ \text{lr}_{\min} + \frac{1}{2}(\text{lr}_{\max} - \text{lr}_{\min})(1 + \cos(\pi \cdot \frac{t - T_{\text{warmup}}}{T_{\text{total}} - T_{\text{warmup}}})) & \text{otherwise} \end{cases}$$

Compare training with and without the schedule.

In [ ]:
# EXERCISE 3: Learning rate schedule
# Your code here

## 8. Key Takeaways

1. **Self-attention** lets each token "look at" every other token to decide what's relevant
2. **Causal masking** ensures tokens can only attend to previous positions (for generation)
3. **Multi-head attention** captures multiple types of relationships in parallel
4. A **Transformer block** = Attention + Feed-Forward + Residuals + LayerNorm
5. **GPT is just** stacked Transformer blocks with token + position embeddings
6. **Temperature and top-k** control the randomness of generation

### What We've Built
A complete, working GPT-style language model. The same architecture — just scaled up 10,000x — is what powers ChatGPT.

### Next Up
**Notebook 4: Training and Fine-Tuning LLMs** — How to train on large datasets, fine-tune pre-trained models, and understand instruction tuning and RLHF.